<a href="https://colab.research.google.com/github/shardas06543/Master-s-Thesis/blob/main/Dataset_Distributions_Version_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
from sklearn.model_selection import train_test_split
import duckdb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

Mounted at /content/drive


In [ ]:
master_frame = pd.read_csv('/content/drive/MyDrive/MS_Thesis_Shreeya_Sharda/Master_Dataset.csv')
real_only = master_frame[(master_frame["Data Type"] == "Real")].sample(frac = 1, random_state=42).reset_index(drop = True) #shuffle rows (frac = 1 --> 100% of rows)
synthetic_only = master_frame[(master_frame["Data Type"] != "Real")]

real_only["Ground Truth"].str.strip().value_counts() # total dataset (real only data)

,count
Ground Truth,
Positive,3053
Negative,1137
Neutral,1096


In [ ]:
synthetic_only[["Ground Truth", "Data Type"]].value_counts()

# This is not the finalized counts of data because we are basing that off of the target counts in the train, test, and validate sections


Ground Truth  Data Type
Negative      Zero-shot    1809
              One-shot     1300
              Few-shot     1300
Neutral       One-shot      699
              Few-shot      400
              Zero-shot     191
Negative      Real            1
Name: count, dtype: int64

In [ ]:
raw_data = real_only[['Review', 'Ground Truth', 'Rating']]
labels_stratify = real_only[['Ground Truth']]

train, test = train_test_split(raw_data, stratify=labels_stratify, test_size=0.3, random_state=42)
validate_df = test.sample(frac=0.33, random_state=42)  # Take 1/3 of the test set (30% of whole dataset) to get 10% of the whole dataset for validation
test_df = test.drop(validate_df.index) # Take the remainder of test (after validate is removed), and set it as the new test set

In [ ]:
validate_df["Ground Truth"].str.strip().value_counts() # This is the real data only
rows_to_drop = validate_df[validate_df["Ground Truth"] == "Positive"].sample(n=195).index
validate_new = validate_df.drop(rows_to_drop)
validate_new["Ground Truth"].str.strip().value_counts()
#validate_new.to_csv('/content/drive/MyDrive/MS_Thesis_Shreeya_Sharda/validate_real.csv', index=False)

# For the purposes of being statistically significant, we will reduce the positive samples to 115

,count
Ground Truth,
Positive,115
Neutral,114
Negative,99


In [ ]:
train["Ground Truth"].str.strip().value_counts() # This is real data only
rows_to_drop = train[train["Ground Truth"] == "Positive"].sample(n=1337).index
train_new = train.drop(rows_to_drop)
train_new["Ground Truth"].str.strip().value_counts()
#train_new.to_csv('/content/drive/MyDrive/MS_Thesis_Shreeya_Sharda/Datasets/train_100%_real.csv', index=False)


# For the purposes of being statistically significant, we will reduce the positive samples to 800

,count
Ground Truth,
Positive,800
Negative,796
Neutral,767


In [ ]:
test_df["Ground Truth"].str.strip().value_counts() # This is real data only
rows_to_drop = test_df[test_df["Ground Truth"] == "Positive"].sample(n=356).index
test_new = test_df.drop(rows_to_drop)
test_new["Ground Truth"].str.strip().value_counts()
#test_new.to_csv('/content/drive/MyDrive/MS_Thesis_Shreeya_Sharda/Datasets/test_real.csv', index=False)
# For the purposes of being statistically significant, we will reduce the positive samples to 250.

,count
Ground Truth,
Positive,250
Negative,242
Neutral,215


**Add synthetic data to each minority class in the test dataset of the same amount as real data.**

In [ ]:
synth_neg = int(len(test_new[test_new["Ground Truth"].str.strip() == "Negative"]))
synth_neu = int(len(test_new[test_new["Ground Truth"].str.strip() == "Neutral"]))

negative_zero = synthetic_only[ (synthetic_only['Data Type'] == 'Zero-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg * 0.1))

negative_one = synthetic_only[ (synthetic_only['Data Type'] == 'One-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg * 0.2))

negative_few = synthetic_only[ (synthetic_only['Data Type'] == 'Few-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg * 0.7))

neutral_zero = synthetic_only[ (synthetic_only['Data Type'] == 'Zero-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(synth_neu * 0.1))

neutral_one = synthetic_only[ (synthetic_only['Data Type'] == 'One-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(synth_neu * 0.2))

neutral_few = synthetic_only[ (synthetic_only['Data Type'] == 'Few-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(synth_neu * 0.7))




In [ ]:
# For the synthetic dataset, we chose to add synthetic samples on top of the existing real samples. The number of synthetic samples per class is equivalent to the number of real samples in each class

test_combined = pd.concat([test_new, negative_zero, negative_one, negative_few, neutral_zero, neutral_one, neutral_few])
test_combined["Ground Truth"].str.strip().value_counts()

,count
Ground Truth,
Negative,483
Neutral,429
Positive,250


**Train Distributions for multiclassification analysis**

In [ ]:
total_neg = len(train_new[train_new['Ground Truth'].str.strip() == 'Negative']['Ground Truth'].str.strip())
total_neg

796

In [ ]:
# Split the train dataset, such that 20% is real and 80% is synthetic (for minority classes only). Majority class stays 100% real

synth_neg_needed = int(total_neg * (0.8))

print(synth_neg_needed)

neu_needed = int (len((train_new[train_new["Ground Truth"] == "Neutral"])) * 0.8)

negative_zero = synthetic_only[ (synthetic_only['Data Type'] == 'Zero-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg_needed * 0.1))

negative_one = synthetic_only[ (synthetic_only['Data Type'] == 'One-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg_needed * 0.2))

negative_few = synthetic_only[ (synthetic_only['Data Type'] == 'Few-shot') & (synthetic_only['Ground Truth'] == 'Negative')].sample(n=int(synth_neg_needed * 0.7))

neutral_zero = synthetic_only[ (synthetic_only['Data Type'] == 'Zero-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(neu_needed * 0.1))

neutral_one = synthetic_only[ (synthetic_only['Data Type'] == 'One-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(neu_needed * 0.2))

neutral_few = synthetic_only[ (synthetic_only['Data Type'] == 'Few-shot') & (synthetic_only['Ground Truth'] == 'Neutral')].sample(n=int(neu_needed * 0.7))

636


ValueError: Cannot take a larger sample than population when 'replace=False'

In [ ]:
# Drop the synth samples
rows_to_drop_negative = train_new[train_new["Ground Truth"] == "Negative"].sample(n=synth_neg_needed).index
print(rows_to_drop_negative)


In [ ]:
train_new = train_new.drop(rows_to_drop_negative)

In [ ]:
# Recalculate rows_to_drop_neutral based on the current state of train_new
rows_to_drop_neutral = train_new[train_new["Ground Truth"] == "Neutral"].sample(n=neu_needed, random_state=42).index

train_new = train_new.drop(rows_to_drop_neutral)

In [ ]:
# # Add synthetic samples to the real dataset
train_new = pd.concat([train_new, negative_zero, negative_one, negative_few, neutral_zero, neutral_one, neutral_few])

In [ ]:
train_new["Ground Truth"].str.strip().value_counts()
#train_new["Rating"].value_counts()
train_new[['Ground Truth','Data Type']].value_counts()

In [ ]:
#train_new.to_csv('/content/drive/MyDrive/MS_Thesis_Shreeya_Sharda/Datasets/train_mixed_20%_real.csv', index=False)